In [8]:
import re
import pandas as pd

In [9]:
def parse_msp_stream(filepath):
    meta, peaks, in_peaks = {}, [], False

    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip() == "":
                if meta:
                    yield {"meta": meta, "peaks": peaks}
                meta, peaks, in_peaks = {}, [], False
                continue

            if re.match(r"^Num\s*Peaks", line, re.IGNORECASE):
                in_peaks = True
                continue

            if in_peaks:
                parts = re.split(r"[\s,;]+", line.strip())
                if len(parts) >= 2:
                    try:
                        peaks.append((float(parts[0]), float(parts[1])))
                    except ValueError:
                        pass
            elif ":" in line:
                key, _, val = line.partition(":")
                meta[key.strip()] = val.strip()

        if meta:
            yield {"meta": meta, "peaks": peaks}


In [10]:
def split_library_by_ionmode(msp_file, batch_size=1000, DB="GNPS"):

    n_pos, n_neg, instr_pos, instr_neg, instr_type_pos, instr_type_neg = 0, 0, [], [], [], []



    for rec in parse_msp_stream(msp_file):

        if DB == "GNPS":
            meta, peaks = rec["meta"], rec["peaks"]
            if not peaks:
                continue
            mode_raw = meta.get("IONMODE", meta.get("Ionization mode", "")).upper()
            mode = "P" if mode_raw.startswith("P") else ("N" if mode_raw.startswith("N") else None)
            if mode is None:
                continue


            if mode == "P":
                n_pos += 1
                instr_pos.append(meta["INSTRUMENT"])
                instr_type_pos.append(meta["INSTRUMENTTYPE"])
            else:
                n_neg += 1
                instr_neg.append(meta["INSTRUMENT"])
                instr_type_neg.append(meta["INSTRUMENTTYPE"])

        elif DB == "MassBank":
            meta, peaks = rec["meta"], rec["peaks"]

            if not peaks:
                continue

            mode_raw = meta.get("Ion_mode", meta.get("Ionization mode", "")).upper()
            mode = "P" if mode_raw.startswith("P") else ("N" if mode_raw.startswith("N") else None)
            if mode is None:
                continue

            if mode == "P":
                n_pos += 1
                instr_pos.append(meta["Instrument"])
                instr_type_pos.append(meta["Instrument_type"])
            else:
                n_neg += 1
                instr_neg.append(meta["Instrument"])
                instr_type_neg.append(meta["Instrument_type"])

    instruments_pos = pd.DataFrame(instr_pos, columns=["Instrument"]).value_counts().reset_index()
    instruments_pos.to_csv(DB + "instruments_pos.csv", index=False)

    instruments_neg = pd.DataFrame(instr_neg, columns=["Instrument"]).value_counts().reset_index()
    instruments_neg.to_csv(DB + "instruments_neg.csv", index=False)

    instrument_type_pos = pd.DataFrame(instr_type_pos, columns=["Instrument_type"]).value_counts().reset_index()
    instrument_type_pos.to_csv(DB + "instrument_type_pos.csv", index=False)

    instrument_type_neg = pd.DataFrame(instr_type_neg, columns=["Instrument_type"]).value_counts().reset_index()
    instrument_type_neg.to_csv(DB + "instrument_type_neg.csv", index=False)

    print(f"Done. Positive: {n_pos}, Negative: {n_neg}")

In [11]:
split_library_by_ionmode(msp_file="~/ALL_GNPS_NO_PROPOGATED.msp", batch_size=1000, DB="GNPS")
split_library_by_ionmode(msp_file="~/MassBank_NISTformat.msp", batch_size=1000, DB="MassBank")

Done. Positive: 713370, Negative: 240680
Done. Positive: 97745, Negative: 41261
